(course-alzheimer-openmm-integration)=

# Path A - Module 45: OpenMM Integration

You have prepared, repaired, and parametrized your system. Now, it's time to let the laws of physics take over. **OpenMM** is a high-performance molecular dynamics engine that runs on GPUs. 

In this module, you will learn to bridge the gap between MolSysMT and OpenMM to run real simulations of your Alzheimer's complex.

In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load our solvated and minimized complex
fibril = msm.convert('pdb:2BEG', to_form='molsysmt.MolSys')
msm.build.add_missing_hydrogens(fibril)
peptide = msm.build.build_peptide('KLVFF')
msm.build.add_missing_hydrogens(peptide)
molsys = msm.merge([fibril, peptide])
msm.molecular_mechanics.potential_energy_minimization(molsys)

### 1. Creating the OpenMM Simulation
MolSysMT can generate an `openmm.Simulation` object from a prepared molecular system containing topology and coordinates. The first requested structure initializes the OpenMM context, and the portable CPU platform is used unless you request another installed platform explicitly.

In [ ]:
# Create an OpenMM Simulation object
# Note: This requires OpenMM installed. 
try:
    simulation = msm.convert(molsys, to_form='openmm.Simulation', 
                             forcefield='AMBER14', water_model='TIP3P-FB',
                             temperature='310 K')
    print(f"OpenMM Simulation ready: {type(simulation)}")
except Exception as e:
    print(f"OpenMM conversion failed: {e}")

### 2. Running the Dynamics
Once you have the simulation object, you can run steps of molecular dynamics. Let's run a very short relaxation of 100 steps.

In [ ]:
if 'simulation' in locals():
    print("Running 100 steps of MD...")
    simulation.step(100)
    print("MD steps completed.")

### 3. Fetching the Result back to MolSysMT
After the simulation runs in OpenMM, you can bring the new coordinates back to MolSysMT to use all the analysis tools you have learned.

In [ ]:
if 'simulation' in locals():
    # Update our molsys object with the latest state from the simulation
    msm.set(molsys, coordinates=simulation.context.getState(getPositions=True).getPositions())
    
    # Now we can view the relaxed state
    msm.view(molsys)

--- 

### 🏆 Path A Challenge: The Simulator

1. Take a small peptide system.
2. Convert it to an `openmm.Simulation` using a temperature of **310 Kelvin** (body temperature).
3. Run **500 steps** of MD.
4. Calculate the **RMSD** between the starting structure and the state after 500 steps.

You have completed the full simulation circle! In **Module 46**, we will wrap up this phase by learning how to perform **Geometric Transformations** in 3D space to arrange our components before they enter the simulator.